# Prädiktive Analyse von Windkraftanlagen
Optimierung und Analyse der Leistung von Windkraftanlagen mit maschinellem Lernen




# Original Data

In [39]:
# Importing important Python Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



# for preprocessing
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.manifold import TSNE
from sklearn.tree import plot_tree
from imblearn.combine import SMOTEENN
from collections import Counter
from sklearn.model_selection import learning_curve
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


#  for evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, make_scorer

# models
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import xgboost as xgb



import warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

ImportError: cannot import name 'parse_version' from 'sklearn.utils' (/opt/anaconda3/lib/python3.11/site-packages/sklearn/utils/__init__.py)

## Data Retrieval

In [ ]:
df = pd.read_csv('pred_maint_wind.csv')
df.head()

In [ ]:
df.info

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.shape

In [ ]:
df.columns

## Exploring and Cleaning

In [ ]:
# Check for missing values
print("Missing Values per Column:")
print(df.isna().sum())

# Option 1: Drop columns with significant missing values
df = df.dropna(axis=1, thresh=int(0.8 * len(df)))  # Keep columns with at least 80% non-null values

# Option 2: Impute missing values for numerical columns
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    df[col] = df[col].fillna(df[col].median())  # Replace with median

# Option 3: Impute missing values for categorical columns
for col in df.select_dtypes(include=['object', 'category']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])  # Replace with mode

As we can see we have relatively cleaned dataset as we have not found any missing value.


In [ ]:
print("Anzahl der fehlenden Werte:", df['Recurred'].isna().sum())


In [ ]:
# Plotting production and power over time
plt.figure(figsize=(12, 4))
plt.plot(df['DateTime'], df['WEC: Production kWh'], label='Production kWh')
plt.plot(df['DateTime'], df['WEC: ava. Power'], label='Average Power')
plt.xlabel('DateTime')
plt.ylabel('Value')
plt.title('Production and Power Over Time')
plt.legend()
plt.show()

In [ ]:
# Scatter plot for average wind speed and average rotation
plt.figure(figsize=(12, 4))
sns.scatterplot(data=df, x='WEC: ava. windspeed', y='WEC: ava. Power')
plt.title('Average Wind Speed vs Average Power')
plt.xlabel('Average Wind Speed')
plt.ylabel('Average Power')
plt.show()

- Die durchschnittliche Leistung („WEC: ava. Power“) bleibt konstant, da Windturbinen innerhalb eines festgelegten Leistungsbereichs arbeiten, der durch das Design und die durchschnittlichen Windbedingungen bestimmt wird.
- Die Produktion („WEC: Production kWh“) steigt im Laufe der Zeit aufgrund längerer Betriebszeiten, reduzierter Ausfallzeiten (insgesamt 6 Fälle) oder verbesserter Turbinenutzung, auch wenn die durchschnittliche Leistungsabgabe stabil bleibt.
- Die Drehzahl der Turbine erreicht ihr Maximum bei etwa **14,7**, was wahrscheinlich durch Designbeschränkungen und Sicherheitsmaßnahmen bedingt ist.
- Windgeschwindigkeiten über **8,9** tragen weniger zur Steigerung der Drehzahl bei, wobei Geschwindigkeiten über **8,9** keinen weiteren Einfluss haben. In einigen Fällen führt eine Windgeschwindigkeit über **23** sogar zu einem Rückgang der Drehzahl.
- Dieses Plateau weist auf eine effektive Turbinensteuerung hin, die die Leistung optimiert und gleichzeitig mechanische Schäden bei hohen Windgeschwindigkeiten verhindert.

In [ ]:
# Assuming 'df' is your dataset and it contains columns related to blades
# Select relevant blade columns
blade_columns = ['Blade A temp.', 'Blade B temp.', 'Blade C temp.']  # Update this list if there are more blade-related columns

# Select the subset of the DataFrame containing only these columns
df_blades = df[blade_columns]

# Calculate the correlation matrix for these blade-related columns
correlation_matrix_blades = df_blades.corr()

# Print the correlation matrix
print("Correlation matrix for Blade columns:")
print(correlation_matrix_blades)

# Optional: Visualize the correlation matrix using a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix_blades, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Correlation Matrix for Blade Temperature Columns")
plt.show()

In [ ]:
# Function to normalize data and plot boxplots, including printing the boxplot values
def plot_boxplots_with_scaling(df, feature_type, feature_keywords, exclude_keywords=None):
    """
    Normalizes data and plots boxplots for min, max, and average columns for a specific feature type.
    Prints boxplot statistics.

    :param df: The DataFrame containing the data
    :param feature_type: The type of feature (e.g., 'Power', 'Windspeed')
    :param feature_keywords: A list of keywords to filter the columns (e.g., ['min', 'max', 'ava.'])
    :param exclude_keywords: List of keywords to exclude from the column filtering (e.g., ['reactive']).
    """
    # Filter columns containing the feature keywords and the feature type
    relevant_columns = [col for col in df.columns if feature_type in col and any(keyword in col for keyword in feature_keywords)]

    # Exclude columns with specific keywords, if provided
    if exclude_keywords:
        relevant_columns = [col for col in relevant_columns if not any(keyword in col.lower() for keyword in exclude_keywords)]

    # Ensure there are columns to plot
    if relevant_columns:
        # Normalize the data using MinMaxScaler
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(df[relevant_columns])
        scaled_df = pd.DataFrame(scaled_data, columns=relevant_columns)

        # Compute descriptive statistics for boxplot values
        boxplot_stats = scaled_df.describe()

        # Print boxplot values
        print(f"\nBoxplot values for {feature_type} features ({', '.join(feature_keywords)}):")
        print(boxplot_stats)

        # Plot the boxplot
        plt.figure(figsize=(6, 4))
        sns.boxplot(data=scaled_df)
        plt.xticks(rotation=45)
        plt.title(f'Scaled Box Plots for {feature_type} Features ({", ".join(feature_keywords)})')
        plt.ylabel('Scaled Values (0 to 1)')
        plt.xlabel('Features')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.show()
    else:
        print(f"No relevant columns found for {feature_type} with keywords {feature_keywords}.")

# Define feature types and keywords to look for
feature_types = [
    {'feature_type': 'Power', 'exclude_keywords': ['reactive']},  # Only Power (not reactive)
    {'feature_type': 'reactive Power', 'exclude_keywords': None},  # Reactive Power
    {'feature_type': 'windspeed', 'exclude_keywords': None},  # Windspeed
    {'feature_type': 'Rotation', 'exclude_keywords': None},  # Rotation
]
feature_keywords = ['min', 'max', 'ava.']

# Generate scaled box plots for each feature type
for feature in feature_types:
    plot_boxplots_with_scaling(
        df,
        feature_type=feature['feature_type'],
        feature_keywords=feature_keywords,
        exclude_keywords=feature.get('exclude_keywords')
    )

In [ ]:
# Function to compute and plot correlation matrix for a specific feature type
def plot_correlation_matrix(df, feature_type, exclude_keywords=None):
    """
    Plots a correlation matrix for min, max, and avg columns of a given feature type.

    :param df: DataFrame containing the data
    :param feature_type: The type of feature to filter (e.g., 'Power', 'Reactive Power')
    :param exclude_keywords: List of keywords to exclude from the column filtering (e.g., ['reactive']).
    """
    # Filter columns containing the feature type and min, max, or average
    relevant_columns = [col for col in df.columns if feature_type in col and
                        ('min' in col.lower() or 'max' in col.lower() or 'ava.' in col.lower())]

    # Exclude columns with specific keywords, if provided
    if exclude_keywords:
        relevant_columns = [col for col in relevant_columns if not any(keyword in col.lower() for keyword in exclude_keywords)]

    if relevant_columns:
        # Calculate the correlation matrix
        correlation_matrix = df[relevant_columns].corr()

        # Plot the heatmap
        plt.figure(figsize=(8, 6))
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
        plt.title(f'Correlation Matrix for {feature_type} ')
        plt.xticks(rotation=45)
        plt.yticks(rotation=45)
        plt.show()

        # Print correlation matrix for inspection
        print(f"\nCorrelation Matrix for {feature_type} :")
        print(correlation_matrix)
    else:
        print(f"No relevant columns found for {feature_type}.")

# Generate correlation matrices
plot_correlation_matrix(df, feature_type='Power', exclude_keywords=['reactive'])
plot_correlation_matrix(df, feature_type='reactive Power')
plot_correlation_matrix(df, feature_type='windspeed')
plot_correlation_matrix(df, feature_type='Rotation')

Nach der Analyse der Beschreibung der 12 Spalten und ihrer Korrelationsmatrix werden die folgenden Empfehlungen zur Merkmalsauswahl zur Optimierung des Modells gemacht:

#### Beizubehaltende Merkmale:
1. **Windgeschwindigkeit**: Behalten Sie **`WEC: ava. Windgeschwindigkeit`**, da sie den allgemeinen Trend erfasst und hoch mit der `WEC: max. Windgeschwindigkeit` (Korrelationswert: 0.961) korreliert ist, wodurch sie das repräsentativste Merkmal darstellt.
2. **Rotation**: Behalten Sie **`WEC: ava. Rotation`**, da es das Verhalten der Turbine insgesamt widerspiegelt und eine sehr hohe Korrelation mit der `WEC: max. Rotation` (Korrelationswert: 0.988) aufweist.
3. **Leistung**: Behalten Sie **`WEC: ava. Leistung`**, da sie die allgemeine Leistungsabgabe darstellt und hoch mit sowohl `WEC: max. Leistung` (Korrelationswert: 0.965) als auch `WEC: min. Leistung` (Korrelationswert: 0.945) korreliert.
4. **Reaktive Leistung**: Behalten Sie **`WEC: ava. reaktive Leistung`**, da sie die allgemeinen Trends erfasst und eine hohe Korrelation mit der `WEC: min. reaktive Leistung` (Korrelationswert: 0.952) aufweist.

#### Zu Entfernende Merkmale:
1. **`WEC: min. Windgeschwindigkeit`**: Entfernen, da es eine schlechte Korrelation mit anderen Windgeschwindigkeitsmerkmalen aufweist und einen unrealistischen Ausreißer (`6553.5`) enthält.
2. **`WEC: min. Rotation`**: Entfernen, aufgrund niedriger Korrelationen mit der durchschnittlichen (0.095) und maximalen Rotation (0.088) sowie potenzieller Datenqualitätsprobleme.
3. **`WEC: max. Leistung`** und **`WEC: min. Leistung`**: Entfernen, da sie stark redundant mit `WEC: ava. Leistung` sind und unnötiges Rauschen hinzufügen.
4. **`WEC: max. reaktive Leistung`** und **`WEC: min. reaktive Leistung`**: Entfernen, um Redundanz zu vermeiden, da z.B. die maximale reaktive Leistung eine Korrelation von 0.831 mit der durchschnittlichen Leistung aufweist.

#### Wichtige Gründe:
1. Vereinfacht das Modell, indem Redundanzen vermieden werden (z. B. Entfernung von Merkmalen, die hoch mit einem anderen beibehaltenen Merkmal korreliert sind).
2. Verbessert die Datenqualität, indem Merkmale mit Ausreißern oder schwacher Korrelation ausgeschlossen werden.
3. Konzentriert sich auf Merkmale, die die repräsentativsten Informationen liefern (z. B. durchschnittliche Werte).

#### Nächste Schritte vor der Modellierung:
1. **Ausreißerentfernung**: Beseitigen Sie extreme Werte in den `min`-Spalten für alle Kategorien.
2. **Merkmals-Skalierung**: Wenden Sie eine Skalierung (z. B. MinMaxScaler oder StandardScaler) an, um die Merkmalsbereiche zu normalisieren.
3. **Dimensionsreduktion**: Fahren Sie mit nur den beibehaltenen Merkmalen fort, um Rauschen zu reduzieren und die Modelleffizienz zu verbessern.

In [ ]:
# Function to plot boxplots and print descriptive statistics for inverter temperatures
def plot_inverter_temps_with_stats(df, inverter_columns, system_name):
    """
    Plots boxplots and prints descriptive statistics for a given system's inverter temperatures.

    :param df: The DataFrame containing the data
    :param inverter_columns: List of column names for the inverter temperatures
    :param system_name: Name of the system (e.g., 'Sys 1', 'Sys 2') for labeling
    """
    if not inverter_columns:
        print(f"No columns provided for {system_name}.")
        return

    # Compute descriptive statistics
    stats = df[inverter_columns].describe()
    print(f"\nBoxplot values for {system_name} Inverter Temperatures:")
    print(stats)

    # Plot boxplot
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=df[inverter_columns])
    plt.xticks(rotation=45)
    plt.title(f'{system_name} Inverter Temperatures')
    plt.ylabel('Temperature (°C)')
    plt.xlabel(f'{system_name} Inverter Columns')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Columns for inverter temperatures
sys1_inverter_temps = [
    'Sys 1 inverter 1 cabinet temp.', 'Sys 1 inverter 2 cabinet temp.',
    'Sys 1 inverter 3 cabinet temp.', 'Sys 1 inverter 4 cabinet temp.',
    'Sys 1 inverter 5 cabinet temp.', 'Sys 1 inverter 6 cabinet temp.',
    'Sys 1 inverter 7 cabinet temp.'
]
sys2_inverter_temps = [
    'Sys 2 inverter 1 cabinet temp.', 'Sys 2 inverter 2 cabinet temp.',
    'Sys 2 inverter 3 cabinet temp.', 'Sys 2 inverter 4 cabinet temp.',
    'Sys 2 inverter 5 cabinet temp.', 'Sys 2 inverter 6 cabinet temp.',
    'Sys 2 inverter 7 cabinet temp.'
]

# Plot and print statistics for Sys 1 and Sys 2 inverter temperatures
plot_inverter_temps_with_stats(df, sys1_inverter_temps, "Sys 1")
plot_inverter_temps_with_stats(df, sys2_inverter_temps, "Sys 2")

Diese Boxplots zeigen, dass die Inverter-Temperaturen von Sys 1 konsistent und realistisch sind, mit Mittelwerten von etwa 27°C bis 28°C, Standardabweichungen von ca. 5,7°C bis 6,3°C und maximalen Temperaturen bis zu 45°C, was auf einen normalen Betrieb hinweist.

Im Gegensatz dazu zeigt Sys 2 Anomalien, einschließlich extrem negativer Mittelwerte (z. B. -13,95°C, -49,83°C) und Mindestwerten von -50°C, was wahrscheinlich auf fehlerhafte Sensoren oder inaktive Inverter zurückzuführen ist.

Diese Unregelmäßigkeiten in Sys 2 erfordern eine weitere Untersuchung, um die Zuverlässigkeit der Daten vor der Analyse sicherzustellen.

### Verteilungsanalyse der `Error`-Spalte

Die `Error`-Spalte im Datensatz weist eine **stark unausgewogene Verteilung** auf:

- Die Mehrheit der Instanzen (48.727) entspricht `Error = 0`, was den normalen Betrieb ohne Fehler darstellt.
- Nicht-null Fehlerwerte, die wahrscheinlich auf Störungen oder abnormale Zustände hinweisen, treten viel seltener auf:
  - Der zweitgrößte Fehler (`Error = 246`) kommt **164 Mal** vor, was im Vergleich zum normalen Betrieb deutlich geringer ist.
  - Andere Fehler (z. B. `10`, `20`, `30` usw.) haben Häufigkeiten im Bereich von **1** bis **35**, wobei `Error = 100` nur **einmal** auftritt.

Diese Ungleichverteilung deutet darauf hin, dass der Datensatz stark vom normalen Betriebszustand dominiert wird, was die Vorhersage von Fehlerzuständen (nicht-null Fehler) ohne entsprechende Handhabung erschwert.

### Implikationen der Ungleichverteilung:
1. **Klassenungleichgewicht in der Klassifikation**:
   - Wird diese Spalte als Ziel für die Klassifikation verwendet, könnte das Ungleichgewicht zu einem Modell führen, das dazu neigt, die Mehrheitsklasse (`Error = 0`) vorherzusagen, wodurch Fehlerzustände übersehen werden.

2. **Auswirkungen auf Evaluierungsmetriken**:
   - Die Standardgenauigkeit könnte in diesem Fall irreführend sein, da die Vorhersage von nur `Error = 0` aufgrund der Dominanz der normalen Instanzen eine hohe Genauigkeit erzielen würde.
   - Metriken wie **Präzision**, **Recall** und **F1-Score** sollten bevorzugt werden, um die Modellleistung zu bewerten.

3. **Seltene Fehlerwerte**:
   - Fehler mit sehr niedrigen Häufigkeiten (z. B. `Error = 100` mit nur einer Instanz) bieten möglicherweise nicht genügend Daten für ein aussagekräftiges Modelltraining.
   - Diese seltenen Fehlerarten könnten in eine "Andere Fehler"-Klasse zusammengefasst werden, um das Klassifikationsproblem zu vereinfachen.

### Fazit:
Das extreme Ungleichgewicht in der `Error`-Spalte stellt eine Herausforderung für die Klassifikation dar und erfordert sorgfältige Vorverarbeitung und Modellauswahl, um sicherzustellen, dass die Minderheitsklassen (Fehlerzustände) effektiv identifiziert werden.

## Preparing & Transforming

In [ ]:
# Compute the correlation matrix
correlation_matrix = df.corr()

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(30, 20))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", cbar=True)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Filter correlations related to the feature 'Error'
error_correlations_all = correlation_matrix['Error']

# Set the absolute threshold
threshold = 0.1

# Display all correlations with 'Error'
print("All correlations with 'Error':")
print(error_correlations_all)

# Identify columns with correlation below the threshold of 0.1
low_correlation_columns = error_correlations_all[abs(error_correlations_all) < threshold]

# Display columns with correlation below the threshold
print(f"\nColumns with correlation below the absolute threshold of {threshold} for 'Error':")
print(low_correlation_columns)

# Count columns with correlation below the threshold
low_correlation_count = len(low_correlation_columns)
print(f"\nNumber of columns with correlation below {threshold} for 'Error': {low_correlation_count}")

# Identify columns with correlation above or equal to the threshold of 0.1
high_correlation_columns = error_correlations_all[abs(error_correlations_all) >= threshold]

# Display columns with correlation above or equal to the threshold
print(f"\nColumns with correlation above or equal to the absolute threshold of {threshold} for 'Error':")
print(high_correlation_columns)

# Count columns with correlation above the threshold
high_correlation_count = len(high_correlation_columns)
print(f"\nNumber of columns with correlation above or equal to {threshold} for 'Error': {high_correlation_count}")

In [ ]:
# Assume df is your DataFrame and 'Error' is your target column
# Select only the numeric columns
X_numeric = df.select_dtypes(include=[np.number])

# Handle missing values - filling NaNs with 0 (you could use other methods based on your context)
X_numeric.fillna(0, inplace=True)

# Add constant to the dataset (necessary for VIF calculation)
X_numeric_with_const = add_constant(X_numeric)

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X_numeric_with_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_numeric_with_const.values, i) for i in range(X_numeric_with_const.shape[1])]

# Set a VIF threshold (between 6 and 10) and filter based on it
vif_threshold_lower = 6
vif_threshold_upper = 10
filtered_vif = vif_data[(vif_data["VIF"] >= vif_threshold_lower) & (vif_data["VIF"] <= vif_threshold_upper)]

# Now filter features that are correlated with 'Error' column
error_correlations = X_numeric.corrwith(df['Error']).abs()  # Absolute correlations with 'Error'
filtered_features = filtered_vif[filtered_vif["Feature"].isin(error_correlations[error_correlations > 0].index)]

# Print the filtered features
print(f"Filtered features with VIF between {vif_threshold_lower} and {vif_threshold_upper} and correlated with 'Error':")
print(filtered_features)

- **WEC: max. reactive Power** (VIF: 8,14) und **Inverter std dev** (VIF: 8,20) sind die Merkmale mit moderater bis hoher Multikollinearität (VIF zwischen 6 und 10).

  - Diese Merkmale weisen eine relativ hohe Korrelation mit anderen Prädiktorvariablen auf, was auf eine gewisse Redundanz in den Informationen hinweisen könnte, die sie liefern.
  - Das Vorhandensein hoher Multikollinearität könnte die Fähigkeit des Modells verzerren, die Koeffizienten für diese Merkmale genau zu schätzen, was zu instabilen Vorhersagen führt.

- Es wird empfohlen, **diese Merkmale weiter zu überprüfen**, insbesondere wenn sie hoch mit anderen Merkmalen oder miteinander korreliert sind.
- Sie könnten **diese Merkmale entfernen** oder mit anderen **kombinieren**.

In [ ]:
# List of columns to drop manually based on their correlation with 'Error'
columns_to_drop = [ 'Time',
    'WEC: min. windspeed', 'WEC: max. windspeed',
    'WEC: min. Rotation', 'WEC: max. Rotation',
    'WEC: max. Power', 'WEC: min. Power',
    'WEC: min. reactive Power', 'WEC: max. reactive Power',
    'Sys 2 inverter 5 cabinet temp.',
    'Sys 2 inverter 6 cabinet temp.', 'Sys 2 inverter 7 cabinet temp.',
    'WEC: ava. Nacel position including cable twisting',
    'WEC: Production kWh',
    'WEC: Production minutes',
    'WEC: ava. available P from wind',
    'WEC: ava. available P technical reasons',
    'WEC: ava. Available P force majeure reasons',
    'WEC: ava. Available P force external reasons',
    'WEC: ava. blade angle A',
    'RTU: ava. Setpoint 1',
    'Inverter std dev',
    'Blade A Temp.','Blade B temp.', 'Blade C temp.'
]

# Drop specified columns from the dataset
df_filtered = df.drop(columns=columns_to_drop, errors='ignore')

# Print remaining columns and their count
remaining_columns = df_filtered.columns.tolist()
print(f"Remaining columns: {remaining_columns}")
print(f"Number of remaining columns: {len(remaining_columns)}")


 The cols are dropped based upon either they had high correlation (more than abs 0.9) and very low correlation( below and equal to abs 0.1) as well as anomaly cols **Sys 2 Inverter 6 Cabinet Temp**. Later on in steps Preparation and Transformation we will add means features for cols like sys 1 and sys 2 temps

In [ ]:
# Extract meaningful time-related features
df_filtered['Month'] = df_filtered['DateTime'].dt.month
df_filtered['Week'] = df_filtered['DateTime'].dt.isocalendar().week
df_filtered['DayOfWeek'] = df_filtered['DateTime'].dt.dayofweek  # 0=Monday, 6=Sunday

# Add a seasonal feature with encoding
def assign_season(month):
    if month in [12, 1, 2]:
        return 0  # Winter
    elif month in [3, 4, 5]:
        return 1  # Spring
    elif month in [6, 7, 8]:
        return 2  # Summer
    elif month in [9, 10, 11]:
        return 3  # Autumn

df_filtered['Season'] = df_filtered['Month'].apply(assign_season)

# Display the first few rows to verify
print(df_filtered[['DateTime', 'Month', 'Week', 'DayOfWeek', 'Season']].head())

In [ ]:
# Aggregating system temperatures
df_filtered['Avg sys 1 inverter temp'] = df_filtered[['Sys 1 inverter 1 cabinet temp.', 'Sys 1 inverter 2 cabinet temp.',
                                                'Sys 1 inverter 3 cabinet temp.', 'Sys 1 inverter 4 cabinet temp.',
                                                'Sys 1 inverter 5 cabinet temp.', 'Sys 1 inverter 6 cabinet temp.',
                                                'Sys 1 inverter 7 cabinet temp.']].mean(axis=1)

df_filtered['Avg sys 2 inverter temp'] = df_filtered[['Sys 2 inverter 1 cabinet temp.', 'Sys 2 inverter 2 cabinet temp.',
                                                     'Sys 2 inverter 3 cabinet temp.', 'Sys 2 inverter 4 cabinet temp.']].mean(axis=1)
# Calculate the mean for Rotor temps
df_filtered['Rotor temp mean'] = df_filtered[['Rotor temp. 1', 'Rotor temp. 2']].mean(axis=1)

# Calculate the mean for Stator temps
df_filtered['Stator temp mean'] = df_filtered[['Stator temp. 1', 'Stator temp. 2']].mean(axis=1)

# Calculate the mean for Nacelle ambient temps
df_filtered['Nacelle ambient temp mean'] = df_filtered[['Nacelle ambient temp. 1', 'Nacelle ambient temp. 2']].mean(axis=1)

In [ ]:
# Define the condition for binary classification
df_filtered['Error Binary'] = df_filtered['Error'].apply(lambda x: 0 if x == 0 else 1)

# Check the distribution of the new binary classes
binary_distribution = df_filtered['Error Binary'].value_counts()

# Print the distribution
print("Distribution of 'Error Binary':")
print(binary_distribution)

# Calculate the total number of "Error Occurred" entries
error_total = binary_distribution[1]  # Count of rows where Error_Class_Binary = 1
no_error_total = binary_distribution[0]  # Count of rows where Error_Class_Binary = 0

print(f"\nSummary:")
print(f"Total No Error instances: {no_error_total}")
print(f"Total Error Occurred instances: {error_total}")

In [ ]:
columns_to_process = ['WEC: ava. windspeed', 'WEC: ava. Power', 'WEC: ava. Rotation', 'WEC: ava. reactive Power']


# Winsorize outliers (cap at 1st and 99th percentiles)
for col in columns_to_process:
    lower_bound = np.percentile(df_filtered[col], 1)
    upper_bound = np.percentile(df_filtered[col], 99)
    df_filtered[col] = np.clip(df_filtered[col], lower_bound, upper_bound)

# Apply Min-Max Scaling
scaler = MinMaxScaler()
df_filtered[columns_to_process] = scaler.fit_transform(df_filtered[columns_to_process])

# Display the scaled DataFrame
df_filtered.describe()

In [ ]:
# Drop columns that will not help in prediction or are redundant
cols_to_drop = ['Error', 'DateTime', 'Sys 1 inverter 1 cabinet temp.', 'Sys 1 inverter 2 cabinet temp.',
                                                'Sys 1 inverter 3 cabinet temp.', 'Sys 1 inverter 4 cabinet temp.',
                                                'Sys 1 inverter 5 cabinet temp.', 'Sys 1 inverter 6 cabinet temp.',
                                                'Sys 1 inverter 7 cabinet temp.', 'Sys 2 inverter 1 cabinet temp.', 'Sys 2 inverter 2 cabinet temp.', "Rotor temp. 1",
    "Rotor temp. 2",
    "Stator temp. 1",
    "Stator temp. 2",
    "Nacelle ambient temp. 1",
    "Nacelle ambient temp. 2",
                                                     'Sys 2 inverter 3 cabinet temp.', 'Sys 2 inverter 4 cabinet temp.']  # 'Error' could be dropped if you are creating separate classification tasks.
df_filtered = df_filtered.drop(columns=cols_to_drop, errors='ignore')

# 8. **Final Dataset Inspection**
print(f"Shape of the dataset after feature engineering: {df_filtered.shape}")
print("Columns after feature engineering:")
print(df_filtered.columns.tolist())

In [ ]:
# Scatter plot with different colors for each Error_Class
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_filtered,
    x='WEC: ava. windspeed',  # Replace with an appropriate feature for the x-axis
    y='WEC: ava. Power',      # Replace with an appropriate feature for the y-axis
    hue='Error Binary',        # Class-based coloring
    palette='tab10',          # Use a color palette with enough distinct colors
    style='Error Binary',      # Different markers for each class
    s=50                      # Marker size
)

# Add labels and title
plt.title('Scatter Plot of Error Classes', fontsize=16)
plt.xlabel('Average Windspeed', fontsize=14)
plt.ylabel('Average Power', fontsize=14)
plt.legend(title='Error Class', bbox_to_anchor=(1.05, 1), loc='upper left')  # Legend outside the plot
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## Model Development and Training

In [ ]:
# 1. Split dataset into features (X) and target (y)
X = df_filtered.drop(columns=['Error Binary'])
y = df_filtered['Error Binary']

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [ ]:
# Define a dictionary of models to test
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='mlogloss')  # Configure XGBoost
}

In [ ]:
# Training Section (Fit)
for model_name, model in models.items():
    print(f"\nTraining {model_name}...")

    # Train the model
    model.fit(X_train, y_train)

    # If the model is a Decision Tree, plot the tree structure
    if isinstance(model, DecisionTreeClassifier):
        plt.figure(figsize=(20, 10))
        plot_tree(model, filled=True, feature_names=X_train.columns.tolist(), class_names=['Class 0', 'Class 1'], proportion=True)
        plt.title(f'Visualization of the Decision Tree ({model_name})')
        plt.show()

    # If the model is a Random Forest, visualize the first tree of the forest
    if isinstance(model, RandomForestClassifier):
        # Visualize the first tree of the Random Forest
        plt.figure(figsize=(50, 30))
        plot_tree(model.estimators_[0], filled=True, feature_names=X_train.columns.tolist(), class_names=['Class 0', 'Class 1'], proportion=True)
        plt.title(f'Visualization of the First Tree in Random Forest ({model_name})')
        plt.show()

    # If the model is XGBoost, visualize the first tree
    if isinstance(model, XGBClassifier):
        # Optionally, plot the first tree of the XGBoost model
        plt.figure(figsize=(20, 10))
        xgb.plot_tree(model, num_trees=0)
        plt.title(f'Visualization of the First Tree in XGBoost ({model_name})')
        plt.show()

## Model Validation and Evaluation

In [ ]:
# Evaluation Section
evaluation_results = {}

for model_name, model in models.items():
    print(f"\nEvaluating {model_name}...")

    # Predict on the test set
    y_pred = model.predict(X_test)

    # Compute evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    evaluation_results[model_name] = {'Accuracy': accuracy}

    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Print results
    print(f"Accuracy for {model_name}: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:")
    print(cm)

    # Plot confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
    disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
    plt.title(f"Confusion Matrix for {model_name}")
    plt.show()

# Summary of Results
print("\n--- Evaluation Results ---")
for model_name, metrics in evaluation_results.items():
    print(f"{model_name} - Accuracy: {metrics['Accuracy']:.4f}")


In [ ]:
# Initialize results list to store evaluation metrics for each model
results = []

# Evaluate each model and calculate metrics for both classes
for model_name, model in models.items():
    # Predict on the test set
    y_pred = model.predict(X_test)

    # Calculate metrics for Class 0
    accuracy = accuracy_score(y_test, y_pred)
    precision_0 = precision_score(y_test, y_pred, pos_label=0)
    recall_0 = recall_score(y_test, y_pred, pos_label=0)
    f1_0 = f1_score(y_test, y_pred, pos_label=0)

    results.append({'Model': f"{model_name} (Class 0)", 'Metric': 'Accuracy', 'Value': accuracy})
    results.append({'Model': f"{model_name} (Class 0)", 'Metric': 'Precision', 'Value': precision_0})
    results.append({'Model': f"{model_name} (Class 0)", 'Metric': 'Recall', 'Value': recall_0})
    results.append({'Model': f"{model_name} (Class 0)", 'Metric': 'F1 Score', 'Value': f1_0})

    # Calculate metrics for Class 1
    precision_1 = precision_score(y_test, y_pred, pos_label=1)
    recall_1 = recall_score(y_test, y_pred, pos_label=1)
    f1_1 = f1_score(y_test, y_pred, pos_label=1)

    results.append({'Model': f"{model_name} (Class 1)", 'Metric': 'Accuracy', 'Value': accuracy})  # Accuracy is the same
    results.append({'Model': f"{model_name} (Class 1)", 'Metric': 'Precision', 'Value': precision_1})
    results.append({'Model': f"{model_name} (Class 1)", 'Metric': 'Recall', 'Value': recall_1})
    results.append({'Model': f"{model_name} (Class 1)", 'Metric': 'F1 Score', 'Value': f1_1})

# Convert results to DataFrame
comparison_df = pd.DataFrame(results)

# Plot model performance comparison (all evaluation metrics for both classes)
plt.figure(figsize=(16, 10))
sns.barplot(data=comparison_df, x="Model", y="Value", hue="Metric", palette="Set2", dodge=True)
plt.title("Model Performance Comparison (Evaluation Metrics by Class)", fontsize=16)
plt.ylabel("Score", fontsize=14)
plt.xlabel("Models (Class)", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.legend(title="Metrics", loc="upper right", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Print the metrics table for both classes
print("\nDetailed Metrics for Each Model and Class:")
print(comparison_df)

In [ ]:
# Ensure the models dictionary has only unique models
unique_models = {model_name: model for model_name, model in models.items()}

# Iterate through unique models for evaluation
for model_name, model in unique_models.items():
    print(f"\nEvaluating {model_name}...")

    # Training performance
    y_train_pred = model.predict(X_train)  # Use the specific model object to predict
    train_accuracy = accuracy_score(y_train, y_train_pred)

    # Test performance
    y_test_pred = model.predict(X_test)  # Use the specific model object to predict
    test_accuracy = accuracy_score(y_test, y_test_pred)

    print(f"Training Accuracy for {model_name}: {train_accuracy:.4f}")
    print(f"Test Accuracy for {model_name}: {test_accuracy:.4f}")

In [ ]:
# Iterate through each model to plot learning curves for Recall and F1-Score
for model_name, model in models.items():
    print(f"\nPlotting Learning Curve for {model_name}...")

    # Generate learning curve for Recall
    train_sizes_recall, train_scores_recall, validation_scores_recall = learning_curve(
        model, X_train, y_train, cv=5, scoring=make_scorer(recall_score, pos_label=1)
    )



    # Plot Recall learning curve
    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes_recall, train_scores_recall.mean(axis=1), label='Training Recall', linestyle='--')
    plt.plot(train_sizes_recall, validation_scores_recall.mean(axis=1), label='Validation Recall')
    plt.xlabel('Training Size')
    plt.ylabel('Recall')
    plt.title(f'Recall Learning Curve for {model_name}')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
# Set up Stratified K-Fold cross-validation
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through each model and compute Stratified K-Fold cross-validation scores
for model_name, model in models.items():
    print(f"\nPerforming Stratified Cross-Validation for {model_name}...")

    # Perform cross-validation
    cross_val_scores = cross_val_score(model, X_train, y_train, cv=stratified_kfold, scoring='recall')

    # Print the results for the current model
    print(f"Stratified Cross-Validation Scores for {model_name}: {cross_val_scores}")
    print(f"Mean Stratified Cross-Validation Score for {model_name}: {cross_val_scores.mean():.4f}")

## Hyperparameter Tuning

In [ ]:
# Define hyperparameters for each model
param_grid = {
    'Decision Tree': {
        'max_depth': [5, 10, 15, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'Random Forest': {
        'n_estimators': [100, 150, 200],
        'max_depth': [10, 20, 30, 50],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    },
    'XGBoost': {
        'n_estimators': [50, 100, 150, 200],
        'max_depth': [3, 6, 9, 12],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.7, 1.0],
        'colsample_bytree': [0.7, 1.0],
    },
}

# Define the models
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42),
}

# Initialize a dictionary to store the best model for each
best_models = {}

# Perform hyperparameter tuning using GridSearchCV for each model
for model_name, model in models.items():
    print(f"Performing GridSearchCV for {model_name}...")

    # Initialize GridSearchCV for each model
    grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid[model_name],
    cv=5,
    scoring=make_scorer(recall_score, pos_label=1),  # Use the custom scorer
    verbose=1,
    n_jobs=-1
)

    # Fit GridSearchCV
    grid_search.fit(X_train, y_train)

    # Store the best model for each model name
    best_models[model_name] = grid_search.best_estimator_

    print(f"Best parameters for {model_name}: {grid_search.best_params_}")
    print(f"Best score for {model_name}: {grid_search.best_score_}\n")

# You can now use the best models for evaluation or predictions

In [ ]:
# Function to plot the Decision Tree, Random Forest, and XGBoost trees
def plot_best_models(best_models, X_train, model_names):
    for model_name in model_names:
        model = best_models[model_name]

        # Decision Tree: Plot the tree structure
        if isinstance(model, DecisionTreeClassifier):
            plt.figure(figsize=(20, 10))
            plot_tree(model, filled=True, feature_names=X_train.columns.tolist(), class_names=model.classes_.astype(str), proportion=True)
            plt.title(f'Visualization of the Decision Tree ({model_name})')
            plt.show()

        # Random Forest: Plot one of the trees from the forest
        elif isinstance(model, RandomForestClassifier):
            tree_to_plot = model.estimators_[0]  # Get the first tree in the forest
            plt.figure(figsize=(20, 10))
            plot_tree(tree_to_plot, filled=True, feature_names=X_train.columns.tolist(), class_names=model.classes_.astype(str), proportion=True)
            plt.title(f'Visualization of the First Tree in Random Forest ({model_name})')
            plt.show()

        # XGBoost: Plot one of the trees from XGBoost
        elif isinstance(model, XGBClassifier):
            plt.figure(figsize=(20, 10))
            xgb.plot_tree(model, num_trees=0)  # Plot the first tree
            plt.title(f'Visualization of the First Tree in XGBoost ({model_name})')
            plt.show()

# Assuming best_models contains the best fitted models after hyperparameter tuning
# and X_train is the training dataset
# List of model names used
model_names = ['Decision Tree', 'Random Forest', 'XGBoost']

# Call the function to plot the models
plot_best_models(best_models, X_train, model_names)

## Evaluating and Validating after Tuning

In [ ]:
# Function to evaluate each model and display classification reports, confusion matrices, and a summary
def evaluate_models(best_models, X_test, y_test):
    # Initialize a dictionary to store evaluation results
    evaluation_results = {}

    for model_name, model in best_models.items():
        print(f"\nEvaluating {model_name}...")

        # Predict on the test set
        y_pred = model.predict(X_test)

        # Compute evaluation metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        # Store metrics in results dictionary
        evaluation_results[model_name] = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        }

        # Print Accuracy
        print(f"Accuracy for {model_name}: {accuracy:.4f}")

        # Print classification report
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

        # Compute and display confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        print("\nConfusion Matrix:")
        print(cm)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
        disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
        plt.title(f"Confusion Matrix for {model_name}")
        plt.show()

# Assuming best_models contains the best-fitted models after hyperparameter tuning
# and X_test, y_test are your test dataset
evaluate_models(best_models, X_test, y_test)

In [ ]:
# Initialize results list to store evaluation metrics for each model
results_2 = []

# Evaluate each model and calculate metrics for both classes
for model_name, model in best_models.items():
    # Predict on the test set
    y_pred = model.predict(X_test)

    # Calculate metrics for Class 0
    accuracy = accuracy_score(y_test, y_pred)
    precision_0 = precision_score(y_test, y_pred, pos_label=0)
    recall_0 = recall_score(y_test, y_pred, pos_label=0)
    f1_0 = f1_score(y_test, y_pred, pos_label=0)

    results_2.append({'Model': f"{model_name} (Class 0)", 'Metric': 'Accuracy', 'Value': accuracy})
    results_2.append({'Model': f"{model_name} (Class 0)", 'Metric': 'Precision', 'Value': precision_0})
    results_2.append({'Model': f"{model_name} (Class 0)", 'Metric': 'Recall', 'Value': recall_0})
    results_2.append({'Model': f"{model_name} (Class 0)", 'Metric': 'F1 Score', 'Value': f1_0})

    # Calculate metrics for Class 1
    precision_1 = precision_score(y_test, y_pred, pos_label=1)
    recall_1 = recall_score(y_test, y_pred, pos_label=1)
    f1_1 = f1_score(y_test, y_pred, pos_label=1)

    results_2.append({'Model': f"{model_name} (Class 1)", 'Metric': 'Accuracy', 'Value': accuracy})  # Accuracy is the same
    results_2.append({'Model': f"{model_name} (Class 1)", 'Metric': 'Precision', 'Value': precision_1})
    results_2.append({'Model': f"{model_name} (Class 1)", 'Metric': 'Recall', 'Value': recall_1})
    results_2.append({'Model': f"{model_name} (Class 1)", 'Metric': 'F1 Score', 'Value': f1_1})

# Convert results to DataFrame
comparison_df = pd.DataFrame(results_2)

# Plot model performance comparison (all evaluation metrics for both classes)
plt.figure(figsize=(16, 10))
sns.barplot(data=comparison_df, x="Model", y="Value", hue="Metric", palette="Set2", dodge=True)
plt.title("Model Performance Comparison after Hyperparameter Tuning (Evaluation Metrics by Class)", fontsize=16)
plt.ylabel("Score", fontsize=14)
plt.xlabel("Models (Class)", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.legend(title="Metrics", loc="upper right", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Print the metrics table for both classes
print("\nDetailed Metrics for Each Model and Class after Hyperparameter Tuning:")
print(comparison_df)

In [ ]:
# Iterate through unique best models for evaluation
for model_name, best_model in best_models.items():
    print(f"\nEvaluating {model_name}...")

    # Training performance for Recall (Class 1)
    y_train_pred = best_model.predict(X_train)
    train_recall = recall_score(y_train, y_train_pred, pos_label=1)

    # Test performance for Recall (Class 1)
    y_test_pred = best_model.predict(X_test)
    test_recall = recall_score(y_test, y_test_pred, pos_label=1)

    print(f"Training Recall for {model_name} after Hyperparameter Tuning: {train_recall:.4f}")
    print(f"Test Recall for {model_name} after Hyperparameter Tuning: {test_recall:.4f}")


In [ ]:
# Function to plot Recall vs max_depth for Decision Tree and Random Forest
def plot_max_depth_vs_recall(models, X_train, y_train, X_test, y_test):
    for model_name, best_model in models.items():
        if isinstance(best_model, (DecisionTreeClassifier, RandomForestClassifier)):
            print(f"\nGenerating Recall Curve for {model_name}...")

            # Define max_depth range
            max_depth_range = range(1, 21)
            training_recalls = []
            validation_recalls = []

            for max_depth in max_depth_range:
                # Update model with current max_depth and fit
                model = best_model.set_params(max_depth=max_depth)
                model.fit(X_train, y_train)

                # Calculate Recall for Class 1
                train_recall = recall_score(y_train, model.predict(X_train), pos_label=1)
                val_recall = recall_score(y_test, model.predict(X_test), pos_label=1)

                training_recalls.append(train_recall)
                validation_recalls.append(val_recall)

            # Plot Recall vs max_depth
            plt.figure(figsize=(10, 6))
            plt.plot(max_depth_range, training_recalls, label='Training Recall', linestyle='--', marker='o', color='blue')
            plt.plot(max_depth_range, validation_recalls, label='Validation Recall', marker='o', color='green')
            plt.xlabel('Max Depth')
            plt.ylabel('Recall (Class 1)')
            plt.title(f'Recall vs Max Depth for {model_name}')
            plt.legend(loc='best')
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.show()

# Function to plot Recall vs n_estimators for XGBoost
def plot_n_estimators_vs_recall(models, X_train, y_train, X_test, y_test):
    for model_name, best_model in models.items():
        if isinstance(best_model, XGBClassifier):
            print(f"\nGenerating Recall Curve for {model_name}...")

            # Define n_estimators range
            n_estimators_range = [50, 100, 150, 200, 250]
            training_recalls = []
            validation_recalls = []

            for n_estimators in n_estimators_range:
                # Update model with current n_estimators and fit
                model = best_model.set_params(n_estimators=n_estimators)
                model.fit(X_train, y_train)

                # Calculate Recall for Class 1
                train_recall = recall_score(y_train, model.predict(X_train), pos_label=1)
                val_recall = recall_score(y_test, model.predict(X_test), pos_label=1)

                training_recalls.append(train_recall)
                validation_recalls.append(val_recall)

            # Plot Recall vs n_estimators
            plt.figure(figsize=(10, 6))
            plt.plot(n_estimators_range, training_recalls, label='Training Recall', linestyle='--', marker='o', color='blue')
            plt.plot(n_estimators_range, validation_recalls, label='Validation Recall', marker='o', color='green')
            plt.xlabel('Number of Estimators')
            plt.ylabel('Recall (Class 1)')
            plt.title(f'Recall vs Number of Estimators for {model_name}')
            plt.legend(loc='best')
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.show()

# Call the functions
plot_max_depth_vs_recall(best_models, X_train, y_train, X_test, y_test)
plot_n_estimators_vs_recall(best_models, X_train, y_train, X_test, y_test)

In [ ]:
# Set up Stratified K-Fold cross-validation
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Cross-Validation performance after hyperparameter tuning for each model
for model_name, best_model in best_models.items():
    print(f"\nPerforming Stratified Cross-Validation for {model_name}...")

    # Compute stratified cross-validation scores
    cross_val_scores = cross_val_score(best_model, X_train, y_train, cv=stratified_kfold, scoring='recall')

    # Print the results
    print(f"Stratified Cross-Validation Scores for {model_name}: {cross_val_scores}")
    print(f"Mean Stratified Cross-Validation Score for {model_name}: {cross_val_scores.mean():.4f}")


## Feature Importance

In [ ]:
# Function to plot and print feature importance for each model
def plot_and_print_feature_importance(models, X_train, model_names):
    for model_name in model_names:
        model = models[model_name]

        # Decision Tree and Random Forest
        if isinstance(model, (DecisionTreeClassifier, RandomForestClassifier)):
            feature_importance = model.feature_importances_

            # Create a DataFrame for feature importance
            importance_df = pd.DataFrame({
                'Feature': X_train.columns,
                'Importance': feature_importance
            }).sort_values(by='Importance', ascending=False)

            # Print feature importance values
            print(f"\nFeature Importance for {model_name}:")
            print(importance_df)

            # Plotting
            plt.figure(figsize=(10, 6))
            sns.barplot(x='Importance', y='Feature', data=importance_df)
            plt.title(f'Feature Importance for {model_name}')
            plt.xlabel('Importance')
            plt.ylabel('Features')
            plt.show()

        # XGBoost Model
        elif isinstance(model, XGBClassifier):
            importance = model.feature_importances_
            feature_names = X_train.columns

            # Create a DataFrame for feature importance
            importance_df = pd.DataFrame({
                'Feature': feature_names,
                'Importance': importance
            }).sort_values(by='Importance', ascending=False)

            # Print feature importance values
            print(f"\nFeature Importance for {model_name}:")
            print(importance_df)

            # Plotting
            plt.figure(figsize=(10, 6))
            sns.barplot(x='Importance', y='Feature', data=importance_df)
            plt.title(f'Feature Importance for {model_name}')
            plt.xlabel('Importance')
            plt.ylabel('Features')
            plt.show()

# Assuming best_models contains the best fitted models after hyperparameter tuning
# and X_train is your training dataset
model_names = ['Decision Tree', 'Random Forest', 'XGBoost']

# Call the function to plot and print feature importance for each model
plot_and_print_feature_importance(best_models, X_train, model_names)

In [ ]:
def get_important_features(best_models, X_train, threshold=0.01):
    # Dictionary to hold importance scores
    feature_importance_scores = pd.DataFrame(0, index=X_train.columns, columns=['Decision Tree', 'Random Forest', 'XGBoost'])

    for model_name, model in best_models.items():
        # Decision Tree and Random Forest
        if isinstance(model, (DecisionTreeClassifier, RandomForestClassifier)):
            importance = model.feature_importances_
            feature_importance_scores[model_name] = importance

        # XGBoost
        elif isinstance(model, XGBClassifier):
            importance = model.feature_importances_
            feature_importance_scores[model_name] = importance

    # Average importance scores across models
    feature_importance_scores['Average'] = feature_importance_scores.mean(axis=1)

    # Select features above the threshold
    selected_features = feature_importance_scores[feature_importance_scores['Average'] > threshold].index.tolist()

    print(f"Selected Features:\n{selected_features}")
    return selected_features

# Get important features
selected_features = get_important_features(best_models, X_train)

In [ ]:
# Show class distribution before applying resampling
print("Class distribution before resampling:")
print(Counter(y_train))

In [ ]:
X_train = X_train.astype(np.float32)  # or another consistent type
y_train = y_train.astype(np.int32)  # ensure binary/int labels are int type

In [ ]:
# Scatter plot with different colors for each Error_Class_Binary after resampling
plt.figure(figsize=(8, 6))

# Ensure consistent column names and assign target column to the DataFrame
sns.scatterplot(
    data=pd.DataFrame(X_resampled_smote_enn, columns=X_train.columns).assign(Error_Binary=y_resampled_smote_enn),
    x='WEC: ava. windspeed',  # Replace with an appropriate feature for the x-axis
    y='WEC: ava. Power',      # Replace with an appropriate feature for the y-axis
    hue='Error_Binary',        # Class-based coloring
    palette='tab10',          # Use a color palette with enough distinct colors
    style='Error_Binary',      # Different markers for each class
    s=50                      # Marker size
)

# Add labels and title
plt.title('Scatter Plot of Error Classes After Resampling', fontsize=16)
plt.xlabel('Average Windspeed', fontsize=14)
plt.ylabel('Average Power', fontsize=14)
plt.legend(title='Error Class', bbox_to_anchor=(1.05, 1), loc='upper left')  # Legend outside the plot
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## Model Development and Training with Resampled Data

In [ ]:
# Define the models for training
rsmpl_models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='mlogloss')  # Configure XGBoost
}

In [ ]:
# --- Training Section (Fit) with rsmpl_models ---
for model_name, model in rsmpl_models.items():
    print(f"\nTraining {model_name}...")

    # Train the model on the resampled data
    model.fit(X_resampled_smote_enn, y_resampled_smote_enn)

    # If the model is a Decision Tree, plot the tree structure
    if isinstance(model, DecisionTreeClassifier):
        plt.figure(figsize=(20, 10))
        plot_tree(model, filled=True, feature_names=X_resampled_smote_enn.columns.tolist(),
                  class_names=['Class 0', 'Class 1'], proportion=True)
        plt.title(f'Visualization of the Decision Tree ({model_name}) after SMOTE-ENN resampling')
        plt.show()

    # If the model is a Random Forest, visualize the first tree of the forest
    if isinstance(model, RandomForestClassifier):
        # Visualize the first tree of the Random Forest
        plt.figure(figsize=(50, 30))
        plot_tree(model.estimators_[0], filled=True, feature_names=X_resampled_smote_enn.columns.tolist(),
                  class_names=['Class 0', 'Class 1'], proportion=True)
        plt.title(f'Visualization of the First Tree in Random Forest ({model_name}) after SMOTE-ENN resampling')
        plt.show()

    # If the model is XGBoost, visualize the first tree
    if isinstance(model, XGBClassifier):
        # Optionally, plot the first tree of the XGBoost model
        plt.figure(figsize=(20, 20))
        xgb.plot_tree(model, num_trees=0)  # Plot the first tree
        plt.title(f'Visualization of the First Tree in XGBoost ({model_name}) after SMOTE-ENN resampling')
        plt.show()

## Evaluation and Validation after resampling

In [ ]:
# Evaluate each trained model (after resampling)
for model_name, model in rsmpl_models.items():
    print(f"\nEvaluating {model_name}...")

    # Make predictions on the test set
    y_pred = model.predict(X_test)

    # Compute recall for Class 1
    recall = recall_score(y_test, y_pred, pos_label=1)
    print(f"{model_name} - Test Recall (Class 1): {recall:.4f}")

    # Print other evaluation metrics
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:")
    print(cm)

    # Plot confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
    disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
    plt.title(f"Confusion Matrix for {model_name}")
    plt.show()

In [ ]:
# Initialize results list to store evaluation metrics for each resampled model
results_rsmpl_1 = []

# Evaluate each resampled model and calculate metrics for both classes
for model_name, model in rsmpl_models.items():
    # Predict on the test set
    y_pred = model.predict(X_test)

    # Calculate metrics for Class 0
    accuracy = accuracy_score(y_test, y_pred)
    precision_0 = precision_score(y_test, y_pred, pos_label=0)
    recall_0 = recall_score(y_test, y_pred, pos_label=0)
    f1_0 = f1_score(y_test, y_pred, pos_label=0)

    results_rsmpl_1.append({'Model': f"{model_name} (Class 0)", 'Metric': 'Accuracy', 'Value': accuracy})
    results_rsmpl_1.append({'Model': f"{model_name} (Class 0)", 'Metric': 'Precision', 'Value': precision_0})
    results_rsmpl_1.append({'Model': f"{model_name} (Class 0)", 'Metric': 'Recall', 'Value': recall_0})
    results_rsmpl_1.append({'Model': f"{model_name} (Class 0)", 'Metric': 'F1 Score', 'Value': f1_0})

    # Calculate metrics for Class 1
    precision_1 = precision_score(y_test, y_pred, pos_label=1)
    recall_1 = recall_score(y_test, y_pred, pos_label=1)
    f1_1 = f1_score(y_test, y_pred, pos_label=1)

    results_rsmpl_1.append({'Model': f"{model_name} (Class 1)", 'Metric': 'Accuracy', 'Value': accuracy})  # Accuracy is the same
    results_rsmpl_1.append({'Model': f"{model_name} (Class 1)", 'Metric': 'Precision', 'Value': precision_1})
    results_rsmpl_1.append({'Model': f"{model_name} (Class 1)", 'Metric': 'Recall', 'Value': recall_1})
    results_rsmpl_1.append({'Model': f"{model_name} (Class 1)", 'Metric': 'F1 Score', 'Value': f1_1})

# Convert results to DataFrame
comparison_df_rsmpl = pd.DataFrame(results_rsmpl_1)

# Plot model performance comparison (all evaluation metrics for resampled models by class)
plt.figure(figsize=(16, 10))
sns.barplot(data=comparison_df_rsmpl, x="Model", y="Value", hue="Metric", palette="Set2", dodge=True)
plt.title("Model Performance Comparison (Evaluation Metrics - Resampled Models by Class)", fontsize=16)
plt.ylabel("Score", fontsize=14)
plt.xlabel("Models (Class)", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.legend(title="Metrics", loc="upper right", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Print the metrics table for both classes
print("\nDetailed Metrics for Each Resampled Model and Class:")
print(comparison_df_rsmpl)

In [ ]:
# Ensure the rsmpl_models dictionary has only unique models
unique_rsmpl_models = {model_name: model for model_name, model in rsmpl_models.items()}

# Iterate through unique resampled models for evaluation
for model_name, model in unique_rsmpl_models.items():
    print(f"\nEvaluating {model_name}...")

    # Training performance
    y_train_pred = model.predict(X_train)  # Predict using the model on training data
    train_recall = recall_score(y_train, y_train_pred, pos_label=1)  # Compute training recall for Class 1

    # Test performance
    y_test_pred = model.predict(X_test)  # Predict using the model on test data
    test_recall = recall_score(y_test, y_test_pred, pos_label=1)  # Compute test recall for Class 1

    # Print training and test recall
    print(f"Training Recall for {model_name} (Class 1): {train_recall:.4f}")
    print(f"Test Recall for {model_name} (Class 1): {test_recall:.4f}")

In [ ]:
# Iterate through resampled models to plot learning curves for Recall
for model_name, model in rsmpl_models.items():
    print(f"\nPlotting Learning Curve for {model_name}...")

    # Generate learning curve for Recall (Class 1)
    train_sizes_recall, train_scores_recall, validation_scores_recall = learning_curve(
        model, X_resampled_smote_enn, y_resampled_smote_enn, cv=5, scoring=make_scorer(recall_score, pos_label=1), n_jobs=-1
    )

    # Plot Recall learning curve
    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes_recall, train_scores_recall.mean(axis=1), label='Training Recall', linestyle='--', color='blue')
    plt.plot(train_sizes_recall, validation_scores_recall.mean(axis=1), label='Validation Recall', color='green')
    plt.xlabel('Training Size')
    plt.ylabel('Recall (Class 1)')
    plt.title(f'Recall Learning Curve for {model_name} (After Resampling)')
    plt.legend(loc='best')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

In [ ]:
# Set up Stratified K-Fold cross-validation
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through each resampled model and compute Stratified K-Fold cross-validation scores
for model_name, model in rsmpl_models.items():
    print(f"\nPerforming Stratified Cross-Validation for {model_name}...")

    # Perform cross-validation
    cross_val_scores = cross_val_score(model, X_train, y_train, cv=stratified_kfold, scoring='recall')

    # Print the results for the current model
    print(f"Stratified Cross-Validation Scores for {model_name}: {cross_val_scores}")
    print(f"Mean Stratified Cross-Validation Score for {model_name}: {cross_val_scores.mean():.4f}")


## Hyperparameter Tuning after Resampling data

In [ ]:
# Define hyperparameters for each model
param_grid_rsmpl = {
    'Decision Tree': {
        'max_depth': [5, 10, 15, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'Random Forest': {
        'n_estimators': [100, 150, 200],
        'max_depth': [10, 20, 30, 50],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    },
    'XGBoost': {
        'n_estimators': [50, 100, 150, 200],
        'max_depth': [3, 6, 9, 12],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.7, 1.0],
        'colsample_bytree': [0.7, 1.0],
    },
}

# Define the models
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42),
}

# Initialize a dictionary to store the best model for each
rsmpl_best_models = {}

# Perform hyperparameter tuning using GridSearchCV for each model
for model_name, model in models.items():
    print(f"Performing GridSearchCV for {model_name}...")

    # Initialize GridSearchCV for each model
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid_rsmpl[model_name],
        cv=5,
        scoring=make_scorer(f1_score, pos_label=1),  # F1 score specifically for Class 1
        verbose=1,
        n_jobs=-1
    )

    # Fit GridSearchCV on the resampled data (use the previously resampled data)
    grid_search.fit(X_resampled_smote_enn, y_resampled_smote_enn)

    # Store the best model for each model name
    rsmpl_best_models[model_name] = grid_search.best_estimator_

    print(f"Best parameters for {model_name}: {grid_search.best_params_}")
    print(f"Best F1 score for {model_name}: {grid_search.best_score_}\n")

# The best models are now stored in the `rsmpl_best_models` dictionary for further evaluation or predictions

In [ ]:
# Function to plot the best fitted models (Decision Tree, Random Forest, XGBoost) after hyperparameter tuning and resampling
def plot_best_models_after_resampling(rsmpl_best_models, feature_names, class_names):
    for model_name, model in rsmpl_best_models.items():
        print(f"\nVisualizing {model_name}...")

        # Decision Tree: Plot the tree structure
        if isinstance(model, DecisionTreeClassifier):
            plt.figure(figsize=(20, 10))
            plot_tree(
                model,
                filled=True,
                feature_names=feature_names,
                class_names=class_names,
                proportion=True
            )
            plt.title(f'Visualization of the Decision Tree ({model_name}) after Resampling')
            plt.show()

        # Random Forest: Plot one of the trees from the forest
        elif isinstance(model, RandomForestClassifier):
            tree_to_plot = model.estimators_[0]  # Get the first tree in the forest
            plt.figure(figsize=(20, 10))
            plot_tree(
                tree_to_plot,
                filled=True,
                feature_names=feature_names,
                class_names=class_names,
                proportion=True
            )
            plt.title(f'Visualization of the First Tree in Random Forest ({model_name}) after Resampling')
            plt.show()

        # XGBoost: Plot one of the trees from XGBoost
        elif isinstance(model, XGBClassifier):
            plt.figure(figsize=(20, 10))
            xgb.plot_tree(model, num_trees=0, fmap='', ax=None)  # Plot the first tree
            plt.title(f'Visualization of the First Tree in XGBoost ({model_name}) after Resampling')
            plt.show()

# Assuming rsmpl_best_models contains the best fitted models after hyperparameter tuning and resampling
# and the feature and class names are provided
feature_names = X_resampled_smote_enn.columns.tolist()  # Ensure this matches your dataset
class_names = ["Class 0", "Class 1"]  # Replace with actual class names if available

# Call the function to plot the models
plot_best_models_after_resampling(rsmpl_best_models, feature_names, class_names)

## Evaluating and Validating model after Resampling & Hyperparameter Tuning

In [ ]:
# Initialize results list to store evaluation metrics for each resampled and hyperparameter-tuned model
results_rsmpl_2 = []

# Evaluate each resampled model after hyperparameter tuning
for model_name, model in rsmpl_best_models.items():
    print(f"\nEvaluating {model_name} after Hyperparameter Tuning and Resampling...")

    # Predict on the test set
    y_pred = model.predict(X_test)

    # Compute metrics for Class 0
    accuracy = accuracy_score(y_test, y_pred)
    precision_0 = precision_score(y_test, y_pred, pos_label=0)
    recall_0 = recall_score(y_test, y_pred, pos_label=0)
    f1_0 = f1_score(y_test, y_pred, pos_label=0)

    results_rsmpl_2.append({'Model': f"{model_name} (Class 0)", 'Metric': 'Accuracy', 'Value': accuracy})
    results_rsmpl_2.append({'Model': f"{model_name} (Class 0)", 'Metric': 'Precision', 'Value': precision_0})
    results_rsmpl_2.append({'Model': f"{model_name} (Class 0)", 'Metric': 'Recall', 'Value': recall_0})
    results_rsmpl_2.append({'Model': f"{model_name} (Class 0)", 'Metric': 'F1 Score', 'Value': f1_0})

    # Compute metrics for Class 1
    precision_1 = precision_score(y_test, y_pred, pos_label=1)
    recall_1 = recall_score(y_test, y_pred, pos_label=1)
    f1_1 = f1_score(y_test, y_pred, pos_label=1)

    results_rsmpl_2.append({'Model': f"{model_name} (Class 1)", 'Metric': 'Accuracy', 'Value': accuracy})  # Same accuracy
    results_rsmpl_2.append({'Model': f"{model_name} (Class 1)", 'Metric': 'Precision', 'Value': precision_1})
    results_rsmpl_2.append({'Model': f"{model_name} (Class 1)", 'Metric': 'Recall', 'Value': recall_1})
    results_rsmpl_2.append({'Model': f"{model_name} (Class 1)", 'Metric': 'F1 Score', 'Value': f1_1})

    # Print Classification Report
    print(f"\nClassification Report for {model_name}:")
    print(classification_report(y_test, y_pred))

    # Save the confusion matrix plot for separate generation
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
    disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
    plt.title(f"Confusion Matrix for {model_name}")
    plt.show()

# Convert results to DataFrame
comparison_df_rsmpl = pd.DataFrame(results_rsmpl_2)

# Plot model performance comparison (all evaluation metrics for resampled models by class)
plt.figure(figsize=(16, 10))
sns.barplot(data=comparison_df_rsmpl, x="Model", y="Value", hue="Metric", palette="Set2", dodge=True)
plt.title("Model Performance Comparison (Evaluation Metrics - Resampled & Tuned Models by Class)", fontsize=16)
plt.ylabel("Score", fontsize=14)
plt.xlabel("Models (Class)", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.legend(title="Metrics", loc="upper right", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Print the metrics table for both classes
print("\nDetailed Metrics for Each Resampled & Tuned Model and Class:")
print(comparison_df_rsmpl)

In [ ]:
# Iterate through unique resampled models after hyperparameter tuning for evaluation
for model_name, model in rsmpl_best_models.items():
    print(f"\nEvaluating {model_name} after Hyperparameter Tuning and Resampling...")

    # Predict on the training set
    y_train_pred = model.predict(X_train)  # Predict using the model on training data
    train_f1 = f1_score(y_train, y_train_pred, pos_label=1)  # Compute training F1-Score for Class 1

    # Predict on the test set
    y_test_pred = model.predict(X_test)  # Predict using the model on test data
    test_f1 = f1_score(y_test, y_test_pred, pos_label=1)  # Compute test F1-Score for Class 1

    # Print training and test F1-Score for the current model
    print(f"Training F1-Score for {model_name} (Class 1): {train_f1:.4f}")
    print(f"Test F1-Score for {model_name} (Class 1): {test_f1:.4f}")

In [ ]:
# Function to plot F1-Score vs max_depth for Decision Tree and Random Forest
def plot_max_depth_vs_f1(models, X_train, y_train, X_test, y_test):
    for model_name, best_model in models.items():
        if isinstance(best_model, (DecisionTreeClassifier, RandomForestClassifier)):
            print(f"\nGenerating F1-Score Curve for {model_name}...")

            # Define max_depth range
            max_depth_range = range(1, 21)
            training_f1_scores = []
            validation_f1_scores = []

            for max_depth in max_depth_range:
                # Update model with current max_depth and fit
                model = best_model.set_params(max_depth=max_depth)
                model.fit(X_train, y_train)

                # Calculate F1-Score for Class 1
                train_f1 = f1_score(y_train, model.predict(X_train), pos_label=1)
                val_f1 = f1_score(y_test, model.predict(X_test), pos_label=1)

                training_f1_scores.append(train_f1)
                validation_f1_scores.append(val_f1)

            # Plot F1-Score vs max_depth
            plt.figure(figsize=(10, 6))
            plt.plot(max_depth_range, training_f1_scores, label='Training F1-Score', linestyle='--', marker='o', color='blue')
            plt.plot(max_depth_range, validation_f1_scores, label='Validation F1-Score', marker='o', color='green')
            plt.xlabel('Max Depth')
            plt.ylabel('F1-Score (Class 1)')
            plt.title(f'F1-Score vs Max Depth for {model_name}')
            plt.legend(loc='best')
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.show()

# Function to plot F1-Score vs n_estimators for XGBoost
def plot_n_estimators_vs_f1(models, X_train, y_train, X_test, y_test):
    for model_name, best_model in models.items():
        if isinstance(best_model, XGBClassifier):
            print(f"\nGenerating F1-Score Curve for {model_name}...")

            # Define n_estimators range
            n_estimators_range = [50, 100, 150, 200, 250]
            training_f1_scores = []
            validation_f1_scores = []

            for n_estimators in n_estimators_range:
                # Update model with current n_estimators and fit
                model = best_model.set_params(n_estimators=n_estimators)
                model.fit(X_train, y_train)

                # Calculate F1-Score for Class 1
                train_f1 = f1_score(y_train, model.predict(X_train), pos_label=1)
                val_f1 = f1_score(y_test, model.predict(X_test), pos_label=1)

                training_f1_scores.append(train_f1)
                validation_f1_scores.append(val_f1)

            # Plot F1-Score vs n_estimators
            plt.figure(figsize=(10, 6))
            plt.plot(n_estimators_range, training_f1_scores, label='Training F1-Score', linestyle='--', marker='o', color='blue')
            plt.plot(n_estimators_range, validation_f1_scores, label='Validation F1-Score', marker='o', color='green')
            plt.xlabel('Number of Estimators')
            plt.ylabel('F1-Score (Class 1)')
            plt.title(f'F1-Score vs Number of Estimators for {model_name}')
            plt.legend(loc='best')
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.show()

# Call the functions
plot_max_depth_vs_f1(best_models, X_train, y_train, X_test, y_test)
plot_n_estimators_vs_f1(best_models, X_train, y_train, X_test, y_test)

In [ ]:
# Set up Stratified K-Fold cross-validation after resampling and hyperparameter tuning
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through each resampled model and compute Stratified K-Fold cross-validation scores
for model_name, model in rsmpl_best_models.items():
    print(f"\nPerforming Stratified Cross-Validation for {model_name} after Resampling and Hyperparameter Tuning...")

    # Perform cross-validation using F1-Score for Class 1
    cross_val_scores = cross_val_score(
        model, X_train, y_train,
        cv=stratified_kfold,
        scoring=make_scorer(f1_score, pos_label=1)
    )

    # Print the results for the current model
    print(f"Stratified Cross-Validation F1-Scores for {model_name}: {cross_val_scores}")
    print(f"Mean Stratified Cross-Validation F1-Score for {model_name}: {cross_val_scores.mean():.4f}")

## Feature Importance after Resampling & Hyperparameter Tuning

In [ ]:
# Function to plot and print feature importance for resampled models
def plot_and_print_feature_importance_after_resampling(rsmpl_best_models, X_resampled_smote_enn):
    for model_name, model in rsmpl_best_models.items():
        print(f"\nFeature Importance for {model_name} after Resampling:")

        # Decision Tree: Print and plot feature importance
        if isinstance(model, DecisionTreeClassifier):
            feature_importance = model.feature_importances_

            # Create a DataFrame for feature importance
            importance_df = pd.DataFrame({
                'Feature': X_resampled_smote_enn.columns,
                'Importance': feature_importance
            }).sort_values(by='Importance', ascending=False)

            # Print feature importance
            print(importance_df)

            # Plot feature importance
            plt.figure(figsize=(10, 6))
            sns.barplot(x='Importance', y='Feature', data=importance_df)
            plt.title(f'Feature Importance for {model_name} (Decision Tree) after Resampling')
            plt.xlabel('Importance')
            plt.ylabel('Features')
            plt.show()

        # Random Forest: Print and plot feature importance
        elif isinstance(model, RandomForestClassifier):
            feature_importance = model.feature_importances_

            # Create a DataFrame for feature importance
            importance_df = pd.DataFrame({
                'Feature': X_resampled_smote_enn.columns,
                'Importance': feature_importance
            }).sort_values(by='Importance', ascending=False)

            # Print feature importance
            print(importance_df)

            # Plot feature importance
            plt.figure(figsize=(10, 6))
            sns.barplot(x='Importance', y='Feature', data=importance_df)
            plt.title(f'Feature Importance for {model_name} (Random Forest) after Resampling')
            plt.xlabel('Importance')
            plt.ylabel('Features')
            plt.show()

        # XGBoost: Print and plot feature importance
        elif isinstance(model, XGBClassifier):
            feature_importance = model.feature_importances_
            feature_names = X_resampled_smote_enn.columns

            # Create a DataFrame for feature importance
            importance_df = pd.DataFrame({
                'Feature': feature_names,
                'Importance': feature_importance
            }).sort_values(by='Importance', ascending=False)

            # Print feature importance
            print(importance_df)

            # Plot feature importance
            plt.figure(figsize=(10, 6))
            sns.barplot(x='Importance', y='Feature', data=importance_df)
            plt.title(f'Feature Importance for {model_name} (XGBoost) after Resampling')
            plt.xlabel('Importance')
            plt.ylabel('Features')
            plt.show()


plot_and_print_feature_importance_after_resampling(rsmpl_best_models, X_resampled_smote_enn)